In [214]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [215]:
# read data

df = pd.read_csv('src/liver_disorder.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col
y = y.replace({1: 0, 2: 1})

In [216]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(207, 6)
(138, 6)


In [217]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


print(y_train.shape)
print(X_train.shape)

(207,)
(207, 7)


In [218]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-5

fail_safe = 5000
iterations = 0
learning_rate = 0.08

epsilon = 1e-15 # const dont change

weights = np.zeros(X_train.shape[1])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1

    print(f"Iterations: {iterations} | current_loss: {current_loss}")


Iterations: 1 | current_loss: 0.6931471805599452
Iterations: 2 | current_loss: 0.6914304259299223
Iterations: 3 | current_loss: 0.6897792410705696
Iterations: 4 | current_loss: 0.6881898905015043
Iterations: 5 | current_loss: 0.6866588975950421
Iterations: 6 | current_loss: 0.6851830267211217
Iterations: 7 | current_loss: 0.6837592660794882
Iterations: 8 | current_loss: 0.6823848113454493
Iterations: 9 | current_loss: 0.6810570502096062
Iterations: 10 | current_loss: 0.6797735478552774
Iterations: 11 | current_loss: 0.6785320333886506
Iterations: 12 | current_loss: 0.6773303872148032
Iterations: 13 | current_loss: 0.676166629336471
Iterations: 14 | current_loss: 0.6750389085408107
Iterations: 15 | current_loss: 0.6739454924314787
Iterations: 16 | current_loss: 0.6728847582583719
Iterations: 17 | current_loss: 0.671855184494681
Iterations: 18 | current_loss: 0.6708553431099779
Iterations: 19 | current_loss: 0.6698838924884037
Iterations: 20 | current_loss: 0.6689395709423543
Iterations:

In [219]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.717391304347826


In [220]:
print(f"""
weights: 
    b: {weights[0]}
    BP: {weights[1]}
    Cholesterol: {weights[2]}
    Age: {weights[3]}
    Pregnant: {weights[4]}

""")


weights: 
    b: 0.43064746700051565
    BP: -0.21907984110748988
    Cholesterol: -0.25130100153230056
    Age: -1.0990689714351751
    Pregnant: 1.032988134108239


